In [6]:
"""
PM2.5 Gridded Anomaly Pipeline v5 — India-ASI consistent
=========================================================
Run from:  delhi_ground_atmos_project/week_7/

Changes from v4:
  - Visualization now uses 300x300 grid + gaussian smoothing (sigma=3)
    to match the look of week_5 stagnation anomaly maps.
  - Anomaly computation stays at 0.05 deg (real data resolution).
  - Plotting upsamples via scipy.interpolate.griddata + gaussian_filter.
"""

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import geopandas as gpd
from shapely.ops import unary_union
from matplotlib.path import Path
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter


# =============================================================
# 1. CONFIG
# =============================================================
ERA5_DIR = "../week_5/"
WIND_FILES = [
    ERA5_DIR + "ERA5_WS_daily_2021.nc",
    ERA5_DIR + "ERA5_WS_daily_2022_2024.nc",
]
T2M_FILE  = ERA5_DIR + "era5_daily_t2m_2021_2024.nc"
T925_FILE = ERA5_DIR + "era5_daily_t925hpa_2021_2024.nc"
TP_FILE   = ERA5_DIR + "era5_daily_tp_2021_2024.nc"

STATION_DATA_DIR = "../final_station_data_clean/"
SHAPEFILE = "../ne_10m_admin_1_states_provinces/ne_10m_admin_1_states_provinces.shp"
OUTPUT_DIR = "pm25_anomaly_maps/"

GRID_RES   = 0.05        # computation grid
PLOT_NGRID = 300          # visualization grid (matches week_5)
SMOOTH_SIGMA = 3          # gaussian smoothing sigma (matches week_5)
IDW_POWER  = 2
MIN_STATIONS_PER_DAY = 10

TARGET_MONTHS = [10, 11, 12, 1, 2]
START_DATE = "2021-10-01"
END_DATE   = "2024-02-29"

WS_THRESHOLD = 3.2
TP_THRESHOLD = 0.001

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================
# 2. ALL 26 STATION COORDINATES
# =============================================================
STATION_COORDS = {
    "113":  (28.6530, 77.1470),
    "114":  (28.6812, 77.3025),
    "115":  (28.6080, 77.0310),
    "122":  (28.6360, 77.1980),
    "124":  (28.5640, 77.1822),
    "125":  (28.6760, 77.1340),
    "301":  (28.6470, 77.3157),
    "1420": (28.6900, 77.1813),
    "1421": (28.4988, 77.1631),
    "1422": (28.5810, 77.0590),
    "1423": (28.7302, 77.1693),
    "1424": (28.5798, 77.2337),
    "1425": (28.6113, 77.2374),
    "1426": (28.8555, 77.0940),
    "1427": (28.6100, 76.9780),
    "1428": (28.5300, 77.2700),
    "1429": (28.5660, 77.2490),
    "1430": (28.7157, 77.1100),
    "1431": (28.6300, 77.2800),
    "1432": (28.7200, 77.2700),
    "1434": (28.7028, 77.1632),
    "1435": (28.6690, 77.3150),
    "1560": (28.7900, 77.0380),
    "1561": (28.6860, 77.0530),
    "1562": (28.5350, 77.1870),
    "1563": (28.6356, 77.1492),
}


# =============================================================
# 3. LOAD DELHI BOUNDARY
# =============================================================
print("[geo ] loading Delhi boundary ...")
india = gpd.read_file(SHAPEFILE)
delhi = india[india["name"].str.contains("Delhi", case=False, na=False)].copy()
delhi = delhi.to_crs(epsg=4326)
delhi_geom = unary_union(delhi.geometry)
minx, miny, maxx, maxy = delhi.total_bounds
pad = 0.05
LAT = (miny - pad, maxy + pad)
LON = (minx - pad, maxx + pad)
print(f"       bbox: lat {LAT}, lon {LON}")


# =============================================================
# 4. COMPUTATION GRID (0.05 deg) + PLOT GRID (300x300)
# =============================================================
# Computation grid
lats = np.arange(LAT[0], LAT[1] + 1e-9, GRID_RES)
lons = np.arange(LON[0], LON[1] + 1e-9, GRID_RES)
grid_lon, grid_lat = np.meshgrid(lons, lats)
print(f"[grid] computation: {len(lats)} x {len(lons)} @ {GRID_RES} deg")

# Plot grid (300x300, smooth)
plot_lons = np.linspace(LON[0], LON[1], PLOT_NGRID)
plot_lats = np.linspace(LAT[0], LAT[1], PLOT_NGRID)
plot_lon2d, plot_lat2d = np.meshgrid(plot_lons, plot_lats)
print(f"[grid] visualization: {PLOT_NGRID} x {PLOT_NGRID}")


# =============================================================
# 5. GEOMETRY MASKS (computation + plot grids)
# =============================================================
def geometry_mask(geom, x, y):
    pts  = np.column_stack([x.ravel(), y.ravel()])
    mask = np.zeros(len(pts), dtype=bool)
    polys = list(geom.geoms) if geom.geom_type == "MultiPolygon" else [geom]
    for poly in polys:
        ext    = Path(np.asarray(poly.exterior.coords))
        inside = ext.contains_points(pts)
        for hole in poly.interiors:
            inside &= ~Path(np.asarray(hole.coords)).contains_points(pts)
        mask |= inside
    return mask.reshape(x.shape)

delhi_mask_comp = geometry_mask(delhi_geom, grid_lon, grid_lat)
delhi_mask_plot = geometry_mask(delhi_geom, plot_lon2d, plot_lat2d)


# =============================================================
# 6. COMPUTE STAGNATION FROM ERA5 — DOMAIN-AVERAGE DAILY FLAG
# =============================================================
print("[era5] loading ERA5 data ...")

ds_wind = xr.open_mfdataset(WIND_FILES, combine="by_coords")
ds_t2m  = xr.open_dataset(T2M_FILE)
ds_t925 = xr.open_dataset(T925_FILE)
ds_tp   = xr.open_dataset(TP_FILE)

ds_t2m  = ds_t2m.rename({"lat": "latitude", "lon": "longitude"})
ds_t925 = ds_t925.rename({"lat": "latitude", "lon": "longitude"})
ds_tp   = ds_tp.rename({"lat": "latitude", "lon": "longitude"})
ds_t925 = ds_t925.squeeze("plev", drop=True)

wind_delhi = ds_wind.sel(latitude=slice(LAT[1], LAT[0]),
                         longitude=slice(LON[0], LON[1]))
t2m_delhi  = ds_t2m.sel(latitude=slice(LAT[1], LAT[0]),
                        longitude=slice(LON[0], LON[1]))
t925_delhi = ds_t925.sel(latitude=slice(LAT[1], LAT[0]),
                         longitude=slice(LON[0], LON[1]))
tp_delhi   = ds_tp.sel(latitude=slice(LAT[1], LAT[0]),
                       longitude=slice(LON[0], LON[1]))

wind_delhi = wind_delhi.assign_coords(time=wind_delhi.time.dt.floor("D"))
t2m_delhi  = t2m_delhi.assign_coords(time=t2m_delhi.time.dt.floor("D"))
t925_delhi = t925_delhi.assign_coords(time=t925_delhi.time.dt.floor("D"))
tp_delhi   = tp_delhi.assign_coords(time=tp_delhi.time.dt.floor("D"))

common_start, common_end = "2021-01-01", "2024-12-31"
wind_delhi = wind_delhi.sel(time=slice(common_start, common_end))
t2m_delhi  = t2m_delhi.sel(time=slice(common_start, common_end))
t925_delhi = t925_delhi.sel(time=slice(common_start, common_end))
tp_delhi   = tp_delhi.sel(time=slice(common_start, common_end))

cond_wind      = wind_delhi["ws10m"] < WS_THRESHOLD
cond_inversion = (t925_delhi["t"] - t2m_delhi["2t"]) > 0
cond_dry       = tp_delhi["tp"] < TP_THRESHOLD
stagnant_mask  = cond_wind & cond_inversion & cond_dry

daily_stag_frac = stagnant_mask.mean(dim=["latitude", "longitude"]).compute()
daily_is_stagnant = daily_stag_frac > 0.5

n_total = len(daily_is_stagnant)
n_stag_total = int(daily_is_stagnant.sum().values)
print(f"[era5] domain-avg stagnation: {n_stag_total}/{n_total} days "
      f"({n_stag_total/n_total*100:.1f}%)")


# =============================================================
# 7. LOAD STATION PM2.5
# =============================================================
print("[cpcb] loading station PM2.5 data ...")


def load_daily_pm25_multi(csv_paths):
    dfs = []
    for path in sorted(csv_paths):
        df = pd.read_csv(path, parse_dates=["timestamp"]).set_index("timestamp")
        dfs.append(df)
    combined = pd.concat(dfs).sort_index()
    combined = combined[~combined.index.duplicated(keep="first")]
    col = "PM 2.5" if "PM 2.5" in combined.columns else "PM2.5"
    daily = combined[col].resample("D").mean()
    hourly_count = combined[col].resample("D").count()
    return daily.where(hourly_count >= 18)


series = {}
for station_id in STATION_COORDS:
    pattern = os.path.join(STATION_DATA_DIR, f"station_{station_id}_*", "*.csv")
    matches = glob.glob(pattern)
    if not matches:
        print(f"  [skip] no files for station_{station_id}")
        continue
    series[station_id] = load_daily_pm25_multi(matches)

pm25_df = pd.DataFrame(series).loc[START_DATE:END_DATE]
pm25_df = pm25_df[pm25_df.index.month.isin(TARGET_MONTHS)]
print(f"[cpcb] PM2.5 daily panel: {pm25_df.shape[0]} days x {pm25_df.shape[1]} stations")

st_lats = np.array([STATION_COORDS[sid][0] for sid in pm25_df.columns])
st_lons = np.array([STATION_COORDS[sid][1] for sid in pm25_df.columns])


# =============================================================
# 8. IDW INTERPOLATION
# =============================================================
print("[idw ] interpolating ...")

LAT_KM = 111.0
LON_KM = 111.0 * np.cos(np.deg2rad(28.6))


def idw_day(slats, slons, svals, glat_flat, glon_flat, p=2):
    dlat = (glat_flat[:, None] - slats[None, :]) * LAT_KM
    dlon = (glon_flat[:, None] - slons[None, :]) * LON_KM
    dist = np.sqrt(dlat ** 2 + dlon ** 2)
    coincident = dist < 1e-6
    dist[coincident] = 1e-6
    w = 1.0 / dist ** p
    out = (w * svals[None, :]).sum(axis=1) / w.sum(axis=1)
    if coincident.any():
        r, c = np.where(coincident)
        out[r] = svals[c]
    return out


glat_flat, glon_flat = grid_lat.ravel(), grid_lon.ravel()

days, fields = [], []
for day, row in pm25_df.iterrows():
    vals = row.values.astype(float)
    mask = ~np.isnan(vals)
    if mask.sum() < MIN_STATIONS_PER_DAY:
        continue
    f = idw_day(st_lats[mask], st_lons[mask], vals[mask],
                glat_flat, glon_flat, p=IDW_POWER)
    fields.append(f.reshape(grid_lat.shape))
    days.append(day)

pm25_grid = xr.DataArray(
    np.stack(fields),
    dims=("time", "latitude", "longitude"),
    coords={"time": pd.to_datetime(days), "latitude": lats, "longitude": lons},
    name="pm25",
)
print(f"[idw ] interpolated cube: {pm25_grid.shape}")


# =============================================================
# 9. ALIGN STAGNATION FLAG
# =============================================================
common_times = np.intersect1d(pm25_grid["time"].values,
                               daily_is_stagnant["time"].values)
pm25_grid = pm25_grid.sel(time=common_times)
stag_flag = daily_is_stagnant.sel(time=common_times)
print(f"[stag] aligned: {len(common_times)} common days")


# =============================================================
# 10. COMPUTE CLIMATOLOGY
# =============================================================
print("[clim] computing per-month climatology ...")

climatology = {}
for m in TARGET_MONTHS:
    month_mask = pm25_grid["time"].dt.month == m
    if month_mask.sum().item() == 0:
        continue
    climatology[m] = pm25_grid.sel(time=month_mask).mean("time").values
    n_days = int(month_mask.sum().item())
    domain_mean = np.nanmean(climatology[m][delhi_mask_comp])
    print(f"       month {m:02d}: {n_days} days pooled, "
          f"domain-mean = {domain_mean:.0f} ug/m3")


# =============================================================
# 11. ANOMALY
# =============================================================
def month_anomaly(pm_da, stag_1d, year, month, clim_field):
    sel = (pm_da["time"].dt.year == year) & (pm_da["time"].dt.month == month)
    if sel.sum().item() == 0:
        return None, 0, 0

    pm_sub   = pm_da.sel(time=sel)
    stag_sub = stag_1d.sel(time=sel)

    n_stag = int(stag_sub.sum().values)
    n_total = int(sel.sum().item())

    if n_stag == 0:
        return None, n_stag, n_total

    pm_stag = pm_sub.where(stag_sub).mean("time", skipna=True)
    anom = pm_stag.values - clim_field
    return anom, n_stag, n_total


# =============================================================
# 12. PASS 1: compute all anomalies + global color scale
# =============================================================
print("[pass1] computing anomalies ...")

pairs = []
for y in range(2021, 2025):
    for m in TARGET_MONTHS:
        d = pd.Timestamp(year=y, month=m, day=1)
        if pd.Timestamp(START_DATE) <= d <= pd.Timestamp(END_DATE):
            pairs.append((y, m))

results = []
all_abs_vals = []

for year, month in pairs:
    if month not in climatology:
        results.append((year, month, None, 0, 0))
        continue
    anom, ns, nt = month_anomaly(pm25_grid, stag_flag, year, month,
                                  climatology[month])
    if anom is None:
        results.append((year, month, None, ns, nt))
        continue
    masked = np.where(delhi_mask_comp, anom, np.nan)
    all_abs_vals.append(np.abs(masked[~np.isnan(masked)]))
    results.append((year, month, anom, ns, nt))

if all_abs_vals:
    all_abs = np.concatenate(all_abs_vals)
    GLOBAL_VMAX = float(np.percentile(all_abs, 95))
    GLOBAL_VMAX = max(10, round(GLOBAL_VMAX / 10) * 10)
else:
    GLOBAL_VMAX = 50

print(f"[pass1] global VMAX = {GLOBAL_VMAX} ug/m3")


# =============================================================
# 13. SMOOTH UPSAMPLING FUNCTION (matches week_5 style)
# =============================================================
def upsample_smooth(anom_2d, src_lons, src_lats, dst_lon2d, dst_lat2d, sigma=3):
    """Upsample 0.05 deg anomaly to 300x300 via griddata + gaussian_filter."""
    src_lon2d, src_lat2d = np.meshgrid(src_lons, src_lats)
    points = np.column_stack([src_lon2d.ravel(), src_lat2d.ravel()])
    values = anom_2d.ravel()

    # remove NaN source points
    valid = ~np.isnan(values)
    if valid.sum() < 4:
        return np.full(dst_lon2d.shape, np.nan)

    interp = griddata(points[valid], values[valid],
                       (dst_lon2d, dst_lat2d), method="cubic")

    # fill any NaN from cubic with nearest
    nans = np.isnan(interp)
    if nans.any():
        nearest = griddata(points[valid], values[valid],
                            (dst_lon2d[nans], dst_lat2d[nans]), method="nearest")
        interp[nans] = nearest

    # gaussian smooth
    smoothed = gaussian_filter(interp, sigma=sigma)
    return smoothed


# =============================================================
# 14. PLOTTING — smooth, fixed color scale
# =============================================================
def plot_map(anom_2d, year, month, n_stag, n_total, path, vmax):
    fig, ax = plt.subplots(figsize=(7, 6))

    # Upsample to 300x300 + smooth
    smooth = upsample_smooth(anom_2d, lons, lats,
                              plot_lon2d, plot_lat2d, sigma=SMOOTH_SIGMA)

    # Mask outside Delhi
    plot_data = np.where(delhi_mask_plot, smooth, np.nan)

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    pc = ax.pcolormesh(plot_lons, plot_lats, plot_data,
                       cmap="RdBu_r", norm=norm, shading="auto")
    plt.colorbar(pc, ax=ax, label="PM$_{2.5}$ anomaly (ug/m3)")

    # Delhi boundary
    for geom in delhi.geometry:
        if geom.geom_type == "Polygon":
            xs, ys = geom.exterior.xy
            ax.plot(xs, ys, color="black", linewidth=1.2)
        elif geom.geom_type == "MultiPolygon":
            for poly in geom.geoms:
                xs, ys = poly.exterior.xy
                ax.plot(xs, ys, color="black", linewidth=1.2)

    # Station dots
    ax.scatter(st_lons, st_lats, s=18, c="black",
               edgecolor="white", linewidth=0.5, zorder=5,
               label="CPCB stations")

    ax.set_xlim(LON[0], LON[1])
    ax.set_ylim(LAT[0], LAT[1])
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    label = pd.Timestamp(year=year, month=month, day=1).strftime("%b %Y")
    ax.set_title(
        f"PM$_{{2.5}}$ anomaly (stagnant - climatology) -- {label}\n"
        f"n_stag={n_stag}/{n_total} days  |  IDW (p={IDW_POWER}), "
        f"smoothed",
        fontsize=10,
    )
    ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# =============================================================
# 15. PASS 2: plot
# =============================================================
print(f"\n[pass2] plotting {len(results)} maps (VMAX={GLOBAL_VMAX}) ...")

for year, month, anom, ns, nt in results:
    if anom is None:
        print(f"  [skip] {year}-{month:02d}: n_stag={ns}/{nt}")
        continue
    out_path = os.path.join(OUTPUT_DIR, f"pm25_anomaly_{year}_{month:02d}.png")
    plot_map(anom, year, month, ns, nt, out_path, GLOBAL_VMAX)
    print(f"  [ok  ] {year}-{month:02d}: n_stag={ns}/{nt} -> {out_path}")

# Save gridded cube
nc_path = os.path.join(OUTPUT_DIR, "pm25_grid_daily_0p05deg.nc")
pm25_grid.to_netcdf(nc_path)
print(f"\n[done] gridded cube saved to {nc_path}")
print("[done] pipeline complete.")

[geo ] loading Delhi boundary ...
       bbox: lat (28.382605902840123, 28.932087511129726), lon (76.78368777866804, 77.38753299275338)
[grid] computation: 11 x 13 @ 0.05 deg
[grid] visualization: 300 x 300
[era5] loading ERA5 data ...
[era5] domain-avg stagnation: 174/1461 days (11.9%)
[cpcb] loading station PM2.5 data ...
[cpcb] PM2.5 daily panel: 454 days x 26 stations
[idw ] interpolating ...
[idw ] interpolated cube: (454, 11, 13)
[stag] aligned: 454 common days
[clim] computing per-month climatology ...
       month 10: 93 days pooled, domain-mean = 142 ug/m3
       month 11: 90 days pooled, domain-mean = 284 ug/m3
       month 12: 93 days pooled, domain-mean = 249 ug/m3
       month 01: 93 days pooled, domain-mean = 236 ug/m3
       month 02: 85 days pooled, domain-mean = 165 ug/m3
[pass1] computing anomalies ...
[pass1] global VMAX = 110 ug/m3

[pass2] plotting 15 maps (VMAX=110) ...
  [skip] 2021-10: n_stag=0/31
  [ok  ] 2021-11: n_stag=16/30 -> pm25_anomaly_maps/pm25_anomaly_